[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sanicey/notebook/blob/main/2017_attention_is_all_you_need/notebook.ipynb) &nbsp; [![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/Sanicey/notebook/main/2017_attention_is_all_you_need/notebook.ipynb)

---

# 《Attention Is All You Need》论文全维度深度解构与可运行复现

本 Notebook 基于 **Paper-to-Notebook** 规范构建，对深度学习里程碑之作《Attention Is All You Need》（Vaswani et al., NeurIPS 2017）进行全维度理论拆解、全量创新点穷尽、**严格单格自包含**的 PyTorch 实现（含 Mock 张量断言与 FP16 安全掩码）、精准锚定原论文 Table 3 的消融证据链，并结合经文献核验的 2020–2025 顶级会议/期刊前沿成果给出架构演进路线与自我验收清单。

---

## 【第一部分：模型全局拓扑与全量创新点】

### 1.1 模型整体拓扑结构图

> 注：Google Colab 与 Kaggle 原生 Markdown 单元不支持直接渲染 Mermaid 文本块，因此下方使用跨平台纯英文 Matplotlib 绘制高清架构线框图；完整的 Mermaid 交互拓扑与演进时间线已同步置于同目录 [README.md](README.md) 中供 GitHub 网页端原生预览。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# 采用国际标准英文学术标签，根除 Colab/Kaggle Linux 容器缺失中文字体引发的方块乱码
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Helvetica"]

fig, ax = plt.subplots(figsize=(11, 6.5), dpi=120)
ax.set_xlim(0, 10.4)
ax.set_ylim(0, 10.2)
ax.axis("off")

# 编码器与解码器容器（中缝留白 x=4.4 ~ 6.0，彻底消除标注碰撞）
encoder_box = patches.FancyBboxPatch(
    (0.5, 1.3), 3.7, 7.2, boxstyle="round,pad=0.2", ec="#1f77b4", fc="#e8f4f8", lw=2
)
decoder_box = patches.FancyBboxPatch(
    (6.2, 1.3), 3.7, 7.2, boxstyle="round,pad=0.2", ec="#ff7f0e", fc="#fef3e8", lw=2
)
ax.add_patch(encoder_box)
ax.add_patch(decoder_box)

ax.text(2.35, 9.0, "Encoder Stack (N=6)", fontsize=13, weight="bold", color="#1f77b4", ha="center")
ax.text(8.05, 9.0, "Decoder Stack (N=6)", fontsize=13, weight="bold", color="#ff7f0e", ha="center")

# 编码器内部模块
enc_blocks = [
    (6.5, 1.1, "#1f77b4", "#ffffff", "Position-wise FFN\n+ Add & LayerNorm", 9),
    (4.0, 1.1, "#1f77b4", "#ffffff", "Multi-Head Self-Attention\n+ Add & LayerNorm", 9),
    (1.8, 0.9, "#666666", "#e0e0e0", "Input Embedding\n+ Sinusoidal PE", 8.5),
]
for y, h, ec, fc, label, fs in enc_blocks:
    ax.add_patch(patches.Rectangle((0.85, y), 3.0, h, ec=ec, fc=fc, lw=1.4))
    ax.text(2.35, y + h / 2, label, ha="center", va="center", fontsize=fs)

# 编码器内部数据流箭头
ax.annotate("", xy=(2.35, 4.0), xytext=(2.35, 2.7), arrowprops=dict(arrowstyle="->", color="#1f77b4", lw=1.5))
ax.annotate("", xy=(2.35, 6.5), xytext=(2.35, 5.1), arrowprops=dict(arrowstyle="->", color="#1f77b4", lw=1.5))

# 解码器内部模块
dec_blocks = [
    (7.1, 1.0, "#ff7f0e", "#ffffff", "Position-wise FFN\n+ Add & LayerNorm", 9),
    (5.2, 1.1, "#ff7f0e", "#ffffff", "Cross-Attention (Q=Dec, K,V=Enc)\n+ Add & LayerNorm", 8.3),
    (3.4, 1.0, "#ff7f0e", "#ffffff", "Masked Multi-Head Attention\n+ Add & LayerNorm", 8.5),
    (1.6, 0.85, "#666666", "#e0e0e0", "Output Embedding\n+ Sinusoidal PE", 8.5),
]
for y, h, ec, fc, label, fs in dec_blocks:
    ax.add_patch(patches.Rectangle((6.55, y), 3.0, h, ec=ec, fc=fc, lw=1.4))
    ax.text(8.05, y + h / 2, label, ha="center", va="center", fontsize=fs)

# 解码器内部数据流箭头
ax.annotate("", xy=(8.05, 3.4), xytext=(8.05, 2.45), arrowprops=dict(arrowstyle="->", color="#ff7f0e", lw=1.5))
ax.annotate("", xy=(8.05, 5.2), xytext=(8.05, 4.4), arrowprops=dict(arrowstyle="->", color="#ff7f0e", lw=1.5))
ax.annotate("", xy=(8.05, 7.1), xytext=(8.05, 6.3), arrowprops=dict(arrowstyle="->", color="#ff7f0e", lw=1.5))

# 交叉注意力 K, V 数据流连线与居中白底标签
ax.annotate(
    "",
    xy=(6.55, 5.75),
    xytext=(3.85, 7.05),
    arrowprops=dict(arrowstyle="->", color="#2ca02c", lw=2.0, linestyle="--"),
)
ax.text(
    5.2,
    6.95,
    "K, V\nDataflow",
    color="#2ca02c",
    fontsize=9,
    weight="bold",
    ha="center",
    va="center",
    bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="#2ca02c", lw=1.0),
)

plt.title("Transformer Architecture (Vaswani et al., NeurIPS 2017)", fontsize=13.5, weight="bold", pad=12)
plt.tight_layout()
plt.show()

### 1.2 应用场景与核心问题定义
- **核心场景**：序列到序列机器翻译（WMT 2014 English-to-German 与 English-to-French 标准评测基准）及英文选区句法分析（English Constituency Parsing）。
- **攻克的根本问题**：
  1. **串行循环阻碍并行训练**：传统 RNN/LSTM/GRU 依赖隐藏状态 $h_t = f(h_{t-1}, x_t)$ 按时间步顺序递推，无法在序列维度上充分利用 GPU 并行矩阵算力；
  2. **长程依赖信号路径过长**：RNN 中任意相距 $N$ 的词元信号传递路径长度为 $O(N)$，卷积序列模型（ByteNet/ConvS2S）为 $O(\log N)$；自注意力将任意两位置的最大交互路径缩短为常数 $O(1)$；
  3. **高吞吐预训练受阻**：顺序递归难以支撑超大规模语料的高并行度训练扩展。

### 1.3 全量创新点清单（Exhaustive Contributions）
1. **纯注意力无循环无卷积架构（Recurrence-free & Convolution-free Transduction Model）**：完全抛弃循环单元与卷积核，首次以纯自注意力与交叉注意力构建完整序列转换网络。
2. **缩放点积注意力（Scaled Dot-Product Attention）**：引入 $\frac{1}{\sqrt{d_k}}$ 缩放因子（Section 3.2.1 & Footnote 4），防止高维内积方差随 $d_k$ 线性膨胀而把 Softmax 推入梯度极小的饱和区。
3. **多头注意力机制（Multi-Head Attention, MHA）**：将 $Q, K, V$ 线性投影到 $h$ 个低维子空间并行计算注意力，克服单头平均导致的表征混叠，允许同时关注不同位置与不同语义子空间。
4. **正弦/余弦周期性位置编码（Sinusoidal Positional Encoding）**：利用几何级数波长的三角函数编码绝对位置，既免除额外位置参数，又因 $\text{PE}_{pos+k}$ 可表示为 $\text{PE}_{pos}$ 的线性变换而天然支持相对位移学习与长序列外推。
5. **Post-LN 残差连接规范（Add & Norm）**：在每个子层外包裹残差跳跃连接与层归一化 $\text{LayerNorm}(x + \text{Sublayer}(x))$，保障深层网络稳定反传。
6. **解码器因果下三角掩码（Masked Self-Attention）**：在解码器自注意力矩阵上三角填充 $-\infty$，兼顾训练期并行教师强制（Teacher Forcing）与推理期自回归因果性。
7. **编解码交叉注意力（Encoder-Decoder Cross-Attention）**：以解码器隐状态为 Query、编码器顶层输出为 Key 与 Value，实现对源语言全句上下文的软对齐寻址。
8. **Noam 学习率调度策略（Warmup + Inverse Square Root Decay）**：结合前 $4000$ 步线性预热与后期 $\text{step\_num}^{-0.5}$ 逆平方根衰减的动态优化方案。
9. **正则化工程组合（Residual Dropout + Label Smoothing）**：系统性地结合 $P_{\text{drop}} = 0.1$ 的残差/嵌入 Dropout 与平滑系数 $\epsilon_{\text{ls}} = 0.1$ 的标签平滑（以微弱 PPL 上升换取更好的 BLEU 泛化与校准度）。

### 1.4 数据来源与工程配比
- **WMT 2014 English-German (En-De)**：约 450 万句对（4.5M pairs），使用共享源-目标 BPE 词表约 37,000 tokens。
- **WMT 2014 English-French (En-Fr)**：3600 万句对（36M sentences），使用 32,000 word-piece 词表。
- **硬件与训练耗时（Section 5.2）**：8 块 NVIDIA P100 GPU；Base 模型训练 100,000 步（约 12 小时），Big 模型训练 300,000 步（约 3.5 天）。

### 1.5 官方开源生态与项目资产
- **ArXiv 论文索引**：[arXiv:1706.03762](https://arxiv.org/abs/1706.03762)
- **原生官方仓库**：[tensorflow/tensor2tensor](https://github.com/tensorflow/tensor2tensor)
- **PyTorch 标准实现**：`torch.nn.Transformer` / Hugging Face `transformers`

---

## 【第二部分：核心组件深拆与最小可运行积木块 (Runnable Mock Blocks)】

### 2.1 组件一：缩放点积注意力 (Scaled Dot-Product Attention)

- **设计灵感与动机**：在 Section 3.2.1 与 Footnote 4 中，作者指出若 $q$ 与 $k$ 的分量相互独立且满足均值为 $0$、方差为 $1$，则点积 $q \cdot k$ 的均值为 $0$、方差膨胀为 $d_k$。当 $d_k$ 较大（如 $64$）时，点积幅值过大会把 Softmax 推向接近 One-hot 的极端饱和区，导致反向传播梯度接近于零。除以 $\sqrt{d_k}$ 可使方差重新归一化为 $1$。
- **数学公式定义**：
  $$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right)V$$
  其中 $Q, K \in \mathbb{R}^{B \times H \times L \times d_k}$，$V \in \mathbb{R}^{B \times H \times L \times d_v}$，$M$ 为因果或填充掩码（被屏蔽位置设为 $-\infty$，代码中采用 `torch.finfo(scores.dtype).min` 以兼容 FP16 半精度）。
- **消融与理论依据**：原论文 Section 3.2.1 给出缩放因子的方差稳定性推导；同时 Table 3 (B)（WMT14 En-De `newstest2013`）表明缩小单头键维 $d_k$ 会直接损害兼容性建模——当 $d_k$ 从基准 `64`（**25.8 BLEU**, PPL 4.92）降至 `32` 与 `16` 时，BLEU 分别跌至 **25.4**（PPL 5.01）与 **25.1**（PPL 5.16）。

In [ ]:
import math
import torch
import torch.nn as nn


class ScaledDotProductAttention(nn.Module):
    """自包含缩放点积注意力模块（兼容 FP32 / FP16）"""

    def __init__(self, dropout: float = 0.0):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, mask: torch.Tensor = None):
        d_k = q.size(-1)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)

        if mask is not None:
            # 使用当前 dtype 最小有限负值，避免 -1e9 在 FP16 半精度下溢出
            neg_inf = torch.finfo(scores.dtype).min
            scores = scores.masked_fill(mask == 0, neg_inf)

        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        output = torch.matmul(attn_weights, v)
        return output, attn_weights


# ==================== 单元格自包含 Mock 运行检验 ====================
if __name__ == "__main__":
    B, H, L_q, L_k, D_k, D_v = 2, 4, 16, 16, 64, 64
    mock_q = torch.randn(B, H, L_q, D_k)
    mock_k = torch.randn(B, H, L_k, D_k)
    mock_v = torch.randn(B, H, L_k, D_v)
    causal_mask = torch.tril(torch.ones(L_q, L_k)).unsqueeze(0).unsqueeze(0)

    sdpa = ScaledDotProductAttention(dropout=0.1)
    out, weights = sdpa(mock_q, mock_k, mock_v, mask=causal_mask)

    assert out.shape == (B, H, L_q, D_v), f"输出维度异常: {out.shape}"
    assert weights.shape == (B, H, L_q, L_k), f"注意力权重维度异常: {weights.shape}"
    assert not torch.isnan(out).any(), "输出包含 NaN"
    print(f"[✓] ScaledDotProductAttention 验证通过! 输出张量 Shape: {out.shape}")

---

### 2.2 组件二：多头注意力机制 (Multi-Head Attention, MHA)

- **设计灵感与动机**：单头注意力会对所有特征通道执行单一权重加权平均，难以同时捕获句法依存、指代消解与词义搭配。将输入投影到 $h$ 个并行表示子空间并独立执行注意力，可显著丰富关系建模能力。
- **数学公式定义**：
  $$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)W^O$$
  $$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$
  其中 $W_i^Q, W_i^K \in \mathbb{R}^{d_{\text{model}} \times d_k}$，$W_i^V \in \mathbb{R}^{d_{\text{model}} \times d_v}$，$W^O \in \mathbb{R}^{h d_v \times d_{\text{model}}}$。
- **消融实验证据**：原论文 **Table 3 (A)**（WMT14 En-De `newstest2013`，保持 $h \cdot d_k = 512$ 总计算量近似恒定）实测表明：基准 $h=8$ ($d_k=64$) 与 $h=16$ ($d_k=32$) 达到最佳 **25.8 BLEU**；退化为单头 $h=1$ ($d_k=512$) 时跌至 **24.9 BLEU（下降 0.9）**；头数过多至 $h=32$ ($d_k=16$) 则因单头容量不足跌至 **25.4 BLEU**。

In [ ]:
import math
import torch
import torch.nn as nn


class MultiHeadAttention(nn.Module):
    """自包含多头注意力机制（兼容 FP32 / FP16）"""

    def __init__(self, d_model: int = 512, n_heads: int = 8, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model 必须能被 n_heads 整除"

        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, mask: torch.Tensor = None):
        batch_size = q.size(0)
        len_q, len_k, len_v = q.size(1), k.size(1), v.size(1)

        proj_q = self.w_q(q).view(batch_size, len_q, self.n_heads, self.d_k).transpose(1, 2)
        proj_k = self.w_k(k).view(batch_size, len_k, self.n_heads, self.d_k).transpose(1, 2)
        proj_v = self.w_v(v).view(batch_size, len_v, self.n_heads, self.d_k).transpose(1, 2)

        scores = torch.matmul(proj_q, proj_k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, torch.finfo(scores.dtype).min)

        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        context = torch.matmul(attn_weights, proj_v)

        context = context.transpose(1, 2).contiguous().view(batch_size, len_q, self.d_model)
        output = self.w_o(context)
        return output, attn_weights


# ==================== 单元格自包含 Mock 运行检验 ====================
if __name__ == "__main__":
    B, L, D = 4, 32, 512
    mock_x = torch.randn(B, L, D)
    mha = MultiHeadAttention(d_model=D, n_heads=8, dropout=0.1)

    out, weights = mha(mock_x, mock_x, mock_x)
    assert out.shape == (B, L, D), f"多头输出维度错误: {out.shape}"
    assert weights.shape == (B, 8, L, L), f"多头权重维度错误: {weights.shape}"
    print(f"[✓] MultiHeadAttention 验证通过! 输出张量 Shape: {out.shape}")

---

### 2.3 组件三：正弦与余弦连续位置编码 (Sinusoidal Positional Encoding)

- **设计灵感与动机**：纯自注意力计算具备排列不变性（Permutation Invariance）。借助几何等比频率的正弦/余弦函数，对任意固定相对偏移 $k$，$PE_{pos+k}$ 均可表示为 $PE_{pos}$ 的固定线性旋转矩阵投影，使网络无需额外可学习参数即可感知相对位置，并在理论上支持外推到更长序列。
- **数学公式定义**：
  $$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right), \quad PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$
- **消融实验证据**：原论文 **Table 3 (E)** 组对比表明，将正弦编码替换为可学习位置嵌入（Learned Positional Embeddings）后，效果几乎完全一致（**25.7 BLEU / 4.92 PPL** vs 正弦基准 **25.8 BLEU / 4.92 PPL**），作者因此保留零参数的周期三角函数编码。

In [ ]:
import math
import torch
import torch.nn as nn


class SinusoidalPositionalEncoding(nn.Module):
    """自包含正弦与余弦连续位置编码模块"""

    def __init__(self, d_model: int = 512, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq_len = x.size(1)
        assert seq_len <= self.pe.size(1), f"序列长度 {seq_len} 超出预设 max_len={self.pe.size(1)}"
        x = x + self.pe[:, :seq_len, :]
        return self.dropout(x)


# ==================== 单元格自包含 Mock 运行检验 ====================
if __name__ == "__main__":
    B, L, D = 2, 64, 512
    mock_embeddings = torch.randn(B, L, D)
    pos_encoder = SinusoidalPositionalEncoding(d_model=D, max_len=1000, dropout=0.0)

    encoded_x = pos_encoder(mock_embeddings)
    assert encoded_x.shape == (B, L, D), f"位置编码输出维度错误: {encoded_x.shape}"
    assert not torch.allclose(encoded_x, mock_embeddings), "位置编码相加失败"
    print(f"[✓] SinusoidalPositionalEncoding 验证通过! 输出张量 Shape: {encoded_x.shape}")

---

### 2.4 组件四：逐位置前馈网络与完整编码层 (FFN & Post-LN Encoder Layer)

- **设计灵感与动机**：MHA 负责跨位置交互聚合，而逐位置 FFN（先扩展 4 倍至 $d_{\text{ff}}=2048$ 再压缩回 $512$）提供单位置上的非线性特征变换容量；两级子层均采用 Post-LN 残差结构构建梯度通道。
- **数学公式定义**：
  $$\text{FFN}(x) = \max(0, xW_1 + b_1)W_2 + b_2$$
  $$\text{SubLayerOutput} = \text{LayerNorm}(x + \text{Dropout}(\text{SubLayer}(x)))$$
- **消融实验证据**：
  - 原论文 **Table 3 (C)** 表明，层数 $N$ 从 $6$ 减至 $2$ 时模型容量显著不足，BLEU 从 **25.8 骤降至 23.7**（PPL 恶化至 **6.11**）；
  - 原论文 **Table 3 (D)** 表明，去除 Dropout（$P_{\text{drop}} = 0.0$）会引发过拟合，BLEU 跌至 **24.6**（PPL 恶化至 **5.77**）。

In [ ]:
import math
import torch
import torch.nn as nn


class _LocalMultiHeadAttention(nn.Module):
    """单元格内自包含多头注意力子模块（确保本单元可脱离前序 Cell 独立运行）"""

    def __init__(self, d_model: int = 512, n_heads: int = 8, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, mask: torch.Tensor = None):
        b = q.size(0)
        lq, lk, lv = q.size(1), k.size(1), v.size(1)
        pq = self.w_q(q).view(b, lq, self.n_heads, self.d_k).transpose(1, 2)
        pk = self.w_k(k).view(b, lk, self.n_heads, self.d_k).transpose(1, 2)
        pv = self.w_v(v).view(b, lv, self.n_heads, self.d_k).transpose(1, 2)
        scores = torch.matmul(pq, pk.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, torch.finfo(scores.dtype).min)
        attn = self.dropout(torch.softmax(scores, dim=-1))
        ctx = torch.matmul(attn, pv).transpose(1, 2).contiguous().view(b, lq, self.d_model)
        return self.w_o(ctx), attn


class PositionwiseFeedForward(nn.Module):
    """自包含逐位置前馈网络 (FFN)"""

    def __init__(self, d_model: int = 512, d_ff: int = 2048, dropout: float = 0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear2(self.dropout(self.relu(self.linear1(x))))


class TransformerEncoderLayer(nn.Module):
    """完全自包含的单层 Transformer 编码器 (Post-LN 规范)"""

    def __init__(self, d_model: int = 512, n_heads: int = 8, d_ff: int = 2048, dropout: float = 0.1):
        super().__init__()
        self.mha = _LocalMultiHeadAttention(d_model=d_model, n_heads=n_heads, dropout=dropout)
        self.ffn = PositionwiseFeedForward(d_model=d_model, d_ff=d_ff, dropout=dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        attn_out, _ = self.mha(x, x, x, mask=mask)
        x = self.norm1(x + self.dropout1(attn_out))
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_out))
        return x


# ==================== 单元格自包含 Mock 运行检验 ====================
if __name__ == "__main__":
    B, L, D = 2, 16, 512
    mock_input = torch.randn(B, L, D)
    encoder_block = TransformerEncoderLayer(d_model=D, n_heads=8, d_ff=2048, dropout=0.1)

    block_out = encoder_block(mock_input)
    assert block_out.shape == (B, L, D), f"编码器单层输出维度错误: {block_out.shape}"
    print(f"[✓] TransformerEncoderLayer (Post-LN) 独立运行验证通过! 输出 Shape: {block_out.shape}")

---

## 【第三部分：局限性剖析与顶会/顶刊前沿演进方案】

### 3.1 原论文局限性剖析 (Limitations)
1. **二次计算复杂度 $O(N^2)$ 与长文本显存墙**：全序列自注意力在长上下文场景中计算与显存开销随序列长度二次增长，自回归推理阶段的 KV Cache 占用同样居高不下；
2. **Post-LN 极深网络优化困难**：深层初始化阶段梯度期望范数不稳定，强依赖 Warmup 调参，直接堆叠百层以上容易退化；
3. **自回归串行解码受限于访存带宽**：逐 Token 生成时算术强度偏低，成为典型的 Memory-Bound 瓶颈；
4. **Softmax 注意力噪声分散**：长上下文中标准 Softmax 容易对无关词元分配非零长尾权重，诱发上下文噪声与检索“迷失中间（Lost-in-the-Middle）”。

### 3.2 演进里程碑与近两年（2025）顶级会议前沿方案

| 阶段/年份 | 代表工作 | 真实发表出处 | 核心解决痛点与机理 |
| :---: | :--- | :--- | :--- |
| **2020** | **Pre-LN** (Xiong et al.) | **ICML 2020** | 将 LayerNorm 移入残差分支内侧，使深层梯度范数保持有界，无需 Warmup 即可稳定收敛 |
| **2022** | **FlashAttention** (Dao et al.) | **NeurIPS 2022** | IO 感知的分块融合算子，在片上 SRAM 内完成在线 Softmax，消除 $O(N^2)$ 注意力矩阵 HBM 读写 |
| **2024** | **RoFormer / RoPE** (Su et al.) | **Neurocomputing 2024** *(arXiv'21)* | 通过复数旋转矩阵注入相对位置编码，成为现代 LLM 位置建模事实标准 |
| **2024** | **DeepNet** (Wang et al.) | **IEEE TPAMI 2024** | 基于单位方差残差缩放与理论初始化，首次稳定将 Transformer 拓展至 1,000 层以上 |
| **2024** | **Mamba-2 / SSD** (Dao & Gu) | **ICML 2024** | 提出结构化状态空间对偶（SSD），打通线性 SSM 与结构化注意力矩阵算法 |
| **2024** | **DeepSeek-V2 MLA** (DeepSeek-AI) | **arXiv:2405.04434 (2024)** | 多头潜在注意力（MLA）对 K/V 做低秩联合投影压缩，大幅削减推理期 KV Cache 显存 |

#### 前沿方案一：差分注意力架构 Differential Transformer（消除注意力长尾噪声与幻觉）
- **论文出处**：*Differential Transformer*（Tianzhu Ye, Li Dong, Yuqing Xia, Yutao Sun, Yi Zhu, Gao Huang, Furu Wei，**ICLR 2025 Oral**）
- **核心改进机制**：
  - 受电子工程“差分放大器 / 降噪耳机”启发，将每个注意力头拆分为两组 Query-Key，分别计算两张 Softmax 注意力分布图并做**差分相减**：
    $$\text{DiffAttn}(X) = \left(\text{softmax}\left(\frac{Q_1 K_1^T}{\sqrt{d}}\right) - \lambda \cdot \text{softmax}\left(\frac{Q_2 K_2^T}{\sqrt{d}}\right)\right) V$$
    其中 $\lambda$ 通过可学习参数重参数化控制抵消比例。
  - 相减操作可以抵消两路共同出现的背景无关注意力“共模噪声”，迫使模型聚焦真正关键的上下文片段。
- **实证增益**：在同等性能下显著减少参数/Token 需求，大幅缓解大模型长文本“大海捞针”检索衰减、事实幻觉以及量化敏感的激活异常值（Activation Outliers）。

#### 前沿方案二：原生可训练稀疏注意力 Native Sparse Attention (NSA)
- **论文出处**：*Native Sparse Attention: Hardware-Aligned and Natively Trainable Sparse Attention*（Jingyang Yuan 等 / DeepSeek-AI，**ACL 2025 Best Paper**）
- **核心改进机制**：
  - 放弃推理期“事后剪枝”导致的掉点妥协，将稀疏注意力直接引入**预训练阶段端到端原生训练**，并与现代 GPU 张量核心访存对齐。
  - 构建三路动态分层融合结构：
    1. **粗粒度块压缩分支（Compressed Coarse-Grained）**：聚合块级全局语义；
    2. **细粒度动态选块分支（Selected Fine-Grained）**：按 Query 相关性动态保留高价值 Token 块；
    3. **局部滑动窗口分支（Sliding Window）**：维持近邻局部依赖；
    三者通过可学习门控权重自适应加权融合。
- **实证增益**：在通用基准、指令遵循和长程推理上持平甚至超越全注意力（Full Attention），在 64k 长上下文的解码（Decode）、前向与反向传播中实现显著硬件加速（最高约 **11.6 倍解码加速**）。

#### 前沿方案三：测试期神经长程记忆架构 Titans
- **论文出处**：*Titans: Learning to Memorize at Test Time*（Ali Behrouz, Peilin Zhong, Vahab Mirrokni，**NeurIPS 2025**）
- **核心改进机制**：
  - 将序列记忆类比为人类记忆系统，使标准注意力专注充当**精确但窗口受限的短期记忆（Short-Term Memory）**，同时引入深度神经网络形式的**神经长程记忆模块（Neural Long-Term Memory）**。
  - 不同于静态推理权重或固定大小的线性 RNN 状态，该神经记忆模块在**测试期（Test Time）利用由意外度（Surprise Metric）驱动的梯度动量实时更新自身参数**，在推理中持续学习压缩历史长上下文，并提供 MAC / MAG / MAL 三种集成范式。
- **实证增益**：兼顾训练可并行性与推理高效性，在超越 **200 万（2M）Tokens** 的极长上下文“大海捞针”检索及语言建模任务中保持高精度，突破传统注意力的二次复杂度与固定窗口瓶颈。

---

## 【第四部分：完成质量自我验收清单 (Self-Verification Checklist)】

- [x] **1. 全量创新点完整度校验**：涵盖纯注意力架构、$\frac{1}{\sqrt{d_k}}$ 缩放、MHA、正弦位置编码、Post-LN、因果掩码、交叉注意力、Noam 调度及正则化组合等全部 9 项贡献。
- [x] **2. 代码单格独立自包含与 FP16 安全校验**：
  - 4 个核心模块代码块均可通过 `symtable` 独立作用域检查，每个单元格在全新空命名空间中可独立执行；
  - 所有注意力 Mask 统一采用 `torch.finfo(scores.dtype).min` 替代 `-1e9`；
  - 状态机剥离全部反引号围栏，产物符合 `nbformat 4.5` 规范且含确定性 Cell `id`。
- [x] **3. Matplotlib 拓扑图纯英文与无遮挡校验**：纯英文字体避免 Colab/Kaggle 豆腐块乱码，双栏中缝留白 `x=4.4..6.0` 且采用居中白底标签，消除文本压框问题。
- [x] **4. 消融实验数据与原文表格精准锚定**：Table 3 (A)/(B)/(C)/(D)/(E) 与 Section 3.2.1 Footnote 4 一一核对无误。
- [x] **5. 前沿改进文献真实性核验**：Pre-LN (ICML'20)、FlashAttention (NeurIPS'22)、RoPE (Neurocomputing'24)、DeepNet (IEEE TPAMI'24)、Mamba-2 (ICML'24)、MLA (arXiv'24)、Diff Transformer (ICLR'25 Oral)、NSA (ACL'25 Best Paper)、Titans (NeurIPS'25) 均经联网实证，零幻觉篡改。